# 📈 Logistická regrese na datech diabetu (RandomizedSearchCV)
**Kurz: Data Science / Machine Learning**  
**Sekce: Homework mezi dny 2 a 3 (Klasifikace)**

### Cíl cvičení:
1. Načíst standardizovaný dataset `diabetes_scaled.csv`.
2. Rozdělit data na trénovací a testovací sadu v poměru 70/30 (stratifikovaně).
3. Vytvořit slovník hyperparametrů pro ladění inverzní síly regularizace `C`.
4. Inicializovat model `LogisticRegression` a pomocí `RandomizedSearchCV` najít optimální hyperparametry.
5. Jako optimalizační metriku zvolit stejnou metriku jako v cvičení 1 (F1-Score / Recall).
6. Natrénovat model s nejlepšími parametry a otestovat jeho efektivitu na testovací sadě.
7. Interpretovat váhy modelu a poměr šancí (Odds Ratios).


In [ ]:
# Krok 1: Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## Krok 2: Načtení škálovaných dat
Načteme normalizovaný a škálovaný dataset `diabetes_scaled.csv`.

In [ ]:
df = pd.read_csv('data/diabetes_scaled.csv')
print(f"Rozměry datasetu: {df.shape[0]} řádků, {df.shape[1]} sloupců")

features = [c for c in df.columns if c != 'outcome']
X = df[features].values
y = df['outcome'].values

df.head()


## Krok 3: Rozdělení dat v poměru 70/30 (Stratified Split)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Trénovací sada: {X_train.shape[0]} pacientek (Diabetiček: {sum(y_train == 1)})")
print(f"Testovací sada:  {X_test.shape[0]} pacientek (Diabetiček: {sum(y_test == 1)})")


## Krok 4 & 5: Definice distribuce hyperparametrů (C) a instance modelu
Prozkoumáme spojitý prostor hodnot `C` v logaritmické škále od $10^{-4}$ do $10^3$ pomocí `scipy.stats.loguniform`.

In [ ]:
# Slovník distribucí pro RandomizedSearchCV
param_distributions = {
    'C': loguniform(1e-4, 1e3),
    'penalty': ['l2', 'l1'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [1000]
}

# Instance základního klasifikátoru
lr = LogisticRegression(random_state=42)


## Krok 6: Hledání optimálních hyperparametrů pomocí RandomizedSearchCV
Volíme stejnou metriku jako v cvičení 1: **F1-score** (případně **Recall**), která v medicínském screeningu vyvažuje senzitivitu záchytu a přesnost.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=lr,
    param_distributions=param_distributions,
    n_iter=100,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print(f"Nejlepší F1 skóre v křížové validaci: {random_search.best_score_:.4f}")
print("Optimální nalezené hyperparametry:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")


## Krok 7 & 8: Trénování na nejlepších parametrech a evaluace testovací sady

In [ ]:
best_lr = random_search.best_estimator_

# Predikce na testovací sadě
y_pred = best_lr.predict(X_test)
y_prob = best_lr.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=" * 60)
print("EVALUACE LOGISTICKÉ REGRESE NA TESTOVACÍ SADĚ (70/30)")
print("=" * 60)
print(f"Accuracy (Přesnost):    {acc * 100:.2f} %")
print(f"Recall (Záchyt):        {rec * 100:.2f} %")
print(f"Precision (Preciznost): {prec * 100:.2f} %")
print(f"F1-Score:               {f1:.4f}")
print(f"ROC-AUC:                {auc:.4f}")
print("=" * 60)
print("\nKlasifikační zpráva:")
print(classification_report(y_test, y_pred, target_names=['Zdravá (0)', 'Diabetes (1)']))


## Vizualizace: Matice záměn (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predikce: Zdravá (0)', 'Predikce: Diabetes (1)'],
            yticklabels=['Skutečnost: Zdravá (0)', 'Skutečnost: Diabetes (1)'])
plt.title(f'Matice záměn LogReg (C={best_lr.C:.4f}, penalty={best_lr.penalty})', fontsize=13)
plt.tight_layout()
plt.show()


## Interpretace vah: Odds Ratios (Poměr šancí)
V logistické regresi vyjadřuje koeficient $w_i$ vliv na logaritmus šance (log-odds).
Exponenciála $\exp(w_i)$ představuje **Odds Ratio (OR)** – multiplikační změnu šance na diabetes při nárůstu daného biomarkeru o 1 směrodatnou odchylku.

In [ ]:
df_weights = pd.DataFrame({
    'Feature': features,
    'Coefficient (w)': best_lr.coef_[0],
    'Odds Ratio (exp(w))': np.exp(best_lr.coef_[0])
}).sort_values('Coefficient (w)', ascending=False)

print(df_weights.to_string(index=False))
